In [ ]:
#!pip install sentence_transformers
#!pip install ag2[retrievechat]
#!pip install pyautogen==0.7.2
#https://microsoft.github.io/autogen/0.2/docs/notebooks/agentchat_RetrieveChat/

# Setting base for Retrive Chat

In [1]:
import json
import os

import chromadb

import autogen


# Accepted file formats for that can be stored in
# a vector database instance
from autogen.retrieve_utils import TEXT_FORMATS

config_list = autogen.config_list_from_json("OAI_CONFIG_LIST.json")

assert len(config_list) > 0
print("models to use: ", [config_list[i]["model"] for i in range(len(config_list))])

C:\Users\User\anaconda3\Lib\site-packages\autogen\oai\gemini.py:65: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Patching name='__init__', member=<function BedrockClient.__init__ at 0x00000275C25702C0>, patched=<function function.__call__ at 0x00000275C2570180>
Patching name='_retries', member=5, patched=5
Patching name='cost', member=<function BedrockClient.cost at 0x00000275C25705E0>, patched=<function function.__call__ at 0x00000275C2570860>
Patching name='create', member=<function BedrockClient.create at 0x00000275C2570540>, patched=<function function.__call__ at 0x00000275C2570900>
Patching name='get_usage', member=<function BedrockClient.get_usage at 0x00000275C2570680>, patched=<function function.__call__ at 0x00000275C25709A0>
Patching name='message_retrieval', member=<function BedrockClient.message_retrieval at 0x00000275C2570360>, patched=<function function.__call__ at 0x00000275C2570A40>
Patching name='parse_custom_params', member=<function BedrockClient.parse_custom_params at 0x00000275C2570400>, patched=<function function.__call__ at 0x00000275C2570AE0>
Patching name='parse_params', 

In [2]:
print("Accepted file formats for `docs_path`:")
print(TEXT_FORMATS)

Accepted file formats for `docs_path`:
['txt', 'json', 'csv', 'tsv', 'md', 'html', 'htm', 'rtf', 'rst', 'jsonl', 'log', 'xml', 'yaml', 'yml', 'pdf', 'mdx']


In [3]:
config_list = [
    {
        # Let's choose the Meta's Llama 3.1 model (model names must match Ollama exactly)
        "model": "llama3",
        # We specify the API Type as 'ollama' so it uses the Ollama client class
        "api_type": "ollama",
        "stream": False,
        #"client_host": "http://192.168.0.1:11434",
        "base_url": "http://localhost:11434",

    }
]

In [4]:
#import agentops
#agentops.init("954ecbcb-9555-470b-a2ad-a6999473648b")

In [4]:

from autogen import AssistantAgent

# 1. create an AssistantAgent instance named "assistant"
assistant = AssistantAgent(
    name="assistant",
    system_message="You are a helpful assistant.",
    llm_config={
        "timeout": 600,
        "cache_seed": 42,
        "config_list": config_list,
    },
)


In [5]:
from pypdf import PdfReader

reader = PdfReader("book.pdf")

docs_content = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        docs_content += text + "\n"

print(docs_content[:1000])  # First 1000 characters

The 22
Immutable Laws
of Marketing
Violate Them at Y our Own Risk
Al Ries and Jack Trout
22 Laws of Marketing  10/31/02  12:23 PM  Page 2
Dedicated to the elimination of
myths and misconceptions
from the marketing process
22 Laws of Marketing  10/31/02  12:23 PM  Page 3
Contents
Introduction ix
1. The Law of Leadership 
2. The Law of the Category 
3. The Law of the Mind 
4. The Law of Perception 
5. The Law of Focus 
6. The Law of Exclusivity 
7. The Law of the Ladder 
8. The Law of Duality 
9. The Law of the Opposite 
10. The Law of Division 
11. The Law of Perspective 
12. The Law of Line Extension 
13. The Law of Sacrifice 
14. The Law of Attributes 
15. The Law of Candor 
16. The Law of Singularity 
17. The Law of Unpredictability 
18. The Law of Success 
19. The Law of Failure 
20.The Law of Hype 
21. The Law of Acceleration 
22. The Law of Resources 
Warning 
About the Author
Credits     
Copyright
About the Publisher
Cover
22 Laws of Marketing  10/31/02  12:23 PM  Page 4
Introdu

In [6]:
with open("book.txt", "w", encoding="utf-8") as f:
    f.write(docs_content)

In [7]:

from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

# Define the RetrieveUserProxyAgent instance
ragproxyagent = RetrieveUserProxyAgent(
    name="ragproxyagent",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,
    retrieve_config={
        "task": "qa",
        "docs_path": ["book.txt"],
        "docs_content": docs_content,  # Provide the extracted content directly
        "chunk_token_size": 200,
        "model": config_list[0]["model"],
        "vector_db": "chroma",
        "overwrite": True,
        "get_or_create": True,
    },
    code_execution_config=False,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# Process Starts for Scenario 1 : Generate code based off docstrings w/o human feedback

In [8]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

qa_problem = "summarize the content and use python code to any calculation given in the book"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=qa_problem)
#agentops.end_session("Success")

Trying to create collection.


max_tokens is too small to fit a single line of text. Breaking this line:
	The 22 ...
Failed to split docs with must_break_at_empty_line being True, set to False.
2026-08-28 11:21:10,457 - autogen.agentchat.contrib.retrieve_user_proxy_agent - INFO - Found 215 chunks.
2026-08-28 11:21:10,467 - autogen.agentchat.contrib.vectordb.chromadb - INFO - No content embedding is provided. Will use the VectorDB's embedding function to generate the content embedding.
Model llama3 not found. Using cl100k_base encoding.


VectorDB returns doc_ids:  [['19e7e4c6', '59b0a769', 'd3314526', '689d0693', 'c35e7eea', 'fdfad756', '782430fd', 'ee5e1a09', 'c8aec9f2', 'a44ded10', '006c4b99', 'b3a1f372', 'ca3ddac1', 'a2073511', 'a64f81a6', '5f3a7a0e', 'ec947885', '91f7db4f', 'f35e32d7', 'a4bef4ef']]
Adding content of doc 19e7e4c6 to context.


Model llama3 not found. Using cl100k_base encoding.


Adding content of doc 59b0a769 to context.


...

Adding content of doc 2878f5f1 to context.


Model llama3 not found. Using cl100k_base encoding.


Adding content of doc a7d4f7fa to context.


Model llama3 not found. Using cl100k_base encoding.


ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: summarize the content and use python code to any calculation given in the book

Context is: Acceleration
Successful programs are not built on fads,
they’re built on trends.
120
22 Laws of Marketing  10/31/02  12:23 PM  Page 120
A fad is a wave in the ocean, and a trend is the tide.
A fad gets a lot of hype, and a trend gets very little.
Like a wave, a fad is very visible, but it goes up and
down in a big hurry. Like the tide, a trend is almost
invisible, but it’ s very powerful over the long term.
A fad is a short-term phenomenon that might be
profitable, but a fad doesn’t last long enough to do a
company much good. Furthermore, a company often
tends

In [ ]:
#agentops.end_session("Success")

# Process Starts for Scenario 2 : Answer a question based off docstrings w/o human feedback

In [9]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

qa_problem = "You are a creative head . do many brainstorms"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=qa_problem)

Model llama3 not found. Using cl100k_base encoding.


VectorDB returns doc_ids:  [['dce049fc', '9b56fe8a', '47380280', 'dbeb0252', '238252ee', 'a0e324b9', 'ac1728c9', '91f7db4f', '24bb950f', '92aaa19b', '59b0a769', '2190c4b9', 'cf6b3884', 'b93ff17c', 'dbe1e409', 'cebbfa48', '3c13213d', 'dd2d170b', 'b1b1338c', '8c2722b6']]
Adding content of doc dce049fc to context.


Model llama3 not found. Using cl100k_base encoding.


Adding content of doc 9b56fe8a to context.


Model llama3 not found. Using cl100k_base encoding.


...

Adding content of doc 006c4b99 to context.


Model llama3 not found. Using cl100k_base encoding.


Adding content of doc c35e7eea to context.


Model llama3 not found. Using cl100k_base encoding.


ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: You are a creative head . do many brainstorms

Context is: 22 Laws of Marketing  10/31/02  12:23 PM  Page 89
No.1 brand of jams and jellies. If your name is bad, you
have two choices: change the name or make fun of it.
The one thing you can’t do is to ignore a bad name.
Which is one reason why you won’t find beer brands
like Gablinger’ s, Grolsch, and Gresedieck in your
supermarket today.
“Avis is only No. 2 in rent-a-cars.” So why go with
them? They must try harder. Everybody knew that Avis
was second in rent-a-cars.
So why go with the obvious? Marketing is often a
search for the obvious. Since you can’t change a mind once
it’ s made up, your market

# Process Starts for Scenario 3 : Generate code based off docstrings w/ human feedback

In [10]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

# set `human_input_mode` to be `ALWAYS`, so the agent will ask for human input at every step.
ragproxyagent.human_input_mode = "ALWAYS"
code_problem = "Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. if any coding requires use python and save it as .py"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=code_problem)

Model llama3 not found. Using cl100k_base encoding.


VectorDB returns doc_ids:  [['9b56fe8a', '238252ee', '2190c4b9', '59b0a769', '47380280', 'a0e324b9', 'eba2b0d4', 'fdfad756', '91f7db4f', '24bb950f', '25b6b7de', '19e7e4c6', 'dce049fc', 'a4bef4ef', 'dd2d170b', 'cab764ca', 'c7c7784b', 'f8d6b800', 'efac0b82', 'dca7f094']]
Adding content of doc 9b56fe8a to context.


Model llama3 not found. Using cl100k_base encoding.


Adding content of doc 238252ee to context.


Model llama3 not found. Using cl100k_base encoding.


...

Model llama3 not found. Using cl100k_base encoding.


ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. if any coding requires use python and save it as .py

Context is: If you have a good idea and you’ve picked up this
book with the thought in mind that all you need is a lit-
tle marketing help, this chapter will throw cold water
on that thought.
Even the best idea in the world won’t go very far
without the money to get it off the ground. Inventors,
entrepreneurs, and assorted idea generators seem to
think that all their good ideas need is professional mar-
keting help.
Nothing could be further from the truth. Marketing
is a game fought in the mind of the pr

Replying as ragproxyagent. Provide feedback to assistant. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  give ideas on datascience


ragproxyagent (to assistant):

give ideas on datascience

--------------------------------------------------------------------------------
assistant (to ragproxyagent):

Based on the provided context, here are 5 marketing ideas for an AI and Data Science training institute:

1. **"Own the Word" Campaign**: Identify a key word or phrase related to Data Science and AI (e.g., "Machine Learning Mastery" or "Data Science Insights") and create a campaign to own that word in the minds of potential students and industry professionals. This could involve targeted online advertising, content marketing, and social media efforts.

Python code snippet:
```python
import pandas as pd

# Define the target word and phrases
target_word = "Machine Learning Mastery"
related_phrases = ["Data Science Insights", "AI Training Institute"]

# Create a dataframe to store the campaign's performance metrics
performance_df = pd.DataFrame(columns=["Impressions", "Clicks", "Conversions"])

# Run the campaign and trac

Replying as ragproxyagent. Provide feedback to assistant. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  exit


# Process Starts for Scenario 4: Answer a question based off docstrings w/ human feedback

In [11]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

# set `human_input_mode` to be `ALWAYS`, so the agent will ask for human input at every step.
ragproxyagent.human_input_mode = "ALWAYS"
qa_problem = "Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. "
chat_result = ragproxyagent.initiate_chat(
    assistant, message=ragproxyagent.message_generator, problem=qa_problem
)  # type "exit" to exit the conversation

Model llama3 not found. Using cl100k_base encoding.


VectorDB returns doc_ids:  [['9b56fe8a', '238252ee', '2190c4b9', '47380280', '91f7db4f', 'a0e324b9', 'eba2b0d4', 'dd2d170b', 'f8d6b800', '59b0a769', '903487ed', '25b6b7de', 'a4bef4ef', 'dca7f094', '24bb950f', 'fdfad756', 'f818e6ad', 'cab764ca', '90c94c91', 'b9cd7533']]
Adding content of doc 9b56fe8a to context.


Model llama3 not found. Using cl100k_base encoding.


Adding content of doc 238252ee to context.


Model llama3 not found. Using cl100k_base encoding.


...

Model llama3 not found. Using cl100k_base encoding.


ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. 

Context is: If you have a good idea and you’ve picked up this
book with the thought in mind that all you need is a lit-
tle marketing help, this chapter will throw cold water
on that thought.
Even the best idea in the world won’t go very far
without the money to get it off the ground. Inventors,
entrepreneurs, and assorted idea generators seem to
think that all their good ideas need is professional mar-
keting help.
Nothing could be further from the truth. Marketing
is a game fought in the mind of the prospect. Y ou need
money to get into a mind. And you 

Replying as ragproxyagent. Provide feedback to assistant. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  give ideas on ai and datascience


ragproxyagent (to assistant):

give ideas on ai and datascience

--------------------------------------------------------------------------------
assistant (to ragproxyagent):

Based on the book, here are 5 marketing ideas for an AI and Data Science training institute:

1. **"Unlock the Power of AI" Campaign**: Create a marketing campaign that speaks to the excitement and potential of AI. Highlight how AI can be used to solve real-world problems, improve decision-making, and drive business success. Use eye-catching visuals, testimonials, and success stories to showcase the institute's expertise and the benefits of AI training.
2. **"Data-Driven Insights" Webinar Series**: Develop a series of webinars that focus on the power of data-driven insights in various industries (e.g., healthcare, finance, retail). Invite industry experts to share their experiences and showcase how AI and Data Science can be used to drive business success. Offer a free trial or a limited-time offer to attendees 

Replying as ragproxyagent. Provide feedback to assistant. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  exit
